# Out-of-Core Pipeline – Performance Analysis (CU\_OOC)

This notebook **automatically discovers** all scenes under `benchmark/nsight_gr_results/CU_OOC/` (each subfolder with at least one `*_analysis.yaml`), extracts metrics from Nsight Graphics YAML exports, and, when available, joins preprocess and batch stats CSVs under `CU_OOC/other_metrics`.

**Expected layout per scene:**
- Folder: `CU_OOC/<scene_id>/YAML/range{ring}_{position}_analysis.yaml` (YAML in `<scene_id>/` root is also accepted if there is no `YAML/` subfolder).
- Any file matching `range\d+_\d+_analysis.yaml` is loaded; scenes do not need to be listed manually.

**Optional CSVs:**
- Preprocess: `media/csv/cuda_outofcore/ooc_preprocess_metrics.csv`
- Per-batch stats: `media/csv/cuda_outofcore/ooc_stats_batch.csv`

**Naming convention:** `range{R}_{P}` — ring R, position P (e.g. 5×3 captures).


In [64]:
import os, re, glob, yaml, json, warnings
from pathlib import Path
from collections import defaultdict
from statistics import mean

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})


## 0. Path configuration


In [ ]:
PROJECT_ROOT = Path(os.path.abspath('')).parent.parent
CU_OOC_ROOT = PROJECT_ROOT / 'benchmark' / 'nsight_gr_results' / 'CU_OOC'
RANGE_YAML_RE = re.compile(r'^range(\d+)_(\d+)_analysis\.yaml$', re.IGNORECASE)

# Human-readable labels for known folders (others: "CU_OOC / <name>")
SCENE_LABEL_HINTS = {
    '8wql': 'Molecule 8wql',
    'g10m': '~10M spheres',
    'g50m': '~50M spheres',
    'g100m': '~100M spheres',
    'g500m': '~500M spheres',
}


def _scene_sort_key(name: str):
    m = re.match(r'^g(\d+)m$', name, re.IGNORECASE)
    if m:
        return (0, int(m.group(1)))
    return (1, name.lower())


def _label_for_folder(folder_name: str) -> str:
    return SCENE_LABEL_HINTS.get(folder_name, f'CU_OOC / {folder_name}')


def discover_cu_ooc_scenes(cu_ooc: Path) -> dict:
    """
    Scan CU_OOC: each subdirectory with at least one range*_analysis.yaml.
    Prefer <scene>/YAML/; if missing or empty, use the scene root.
    """
    found = {}
    if not cu_ooc.is_dir():
        return found
    for child in sorted(cu_ooc.iterdir(), key=lambda p: p.name.lower()):
        if not child.is_dir():
            continue
        yaml_dir = child / 'YAML'
        if not yaml_dir.is_dir():
            yaml_dir = child
        elif not any(yaml_dir.glob('*_analysis.yaml')):
            yaml_dir = child
        yfiles = sorted(yaml_dir.glob('*_analysis.yaml'))
        valid = [fp for fp in yfiles if RANGE_YAML_RE.match(fp.name)]
        if not valid:
            continue
        found[child.name] = {
            'label': _label_for_folder(child.name),
            'yaml_dir': yaml_dir,
            'yaml_files': valid,
        }
    return {k: found[k] for k in sorted(found.keys(), key=_scene_sort_key)}


SCENES = discover_cu_ooc_scenes(CU_OOC_ROOT)

OTHER_METRICS_DIR = CU_OOC_ROOT / 'other_metrics'
CSV_BASE_BUILD = PROJECT_ROOT / 'out' / 'build' / 'vs-release' / 'media' / 'csv' / 'cuda_outofcore'
CSV_BASE_FALLBACK = PROJECT_ROOT / 'media' / 'csv' / 'cuda_outofcore'


def _profiler_csv_dir() -> Path:
    if OTHER_METRICS_DIR.is_dir():
        return OTHER_METRICS_DIR
    if CSV_BASE_BUILD.is_dir():
        return CSV_BASE_BUILD
    return CSV_BASE_FALLBACK


PREPROCESS_CSV = _profiler_csv_dir() / 'ooc_preprocess_metrics.csv'
BATCH_CSV_RE_FILE = re.compile(r'^ooc_stats_batch_(.+)\.csv$', re.IGNORECASE)


def discover_batch_csvs() -> dict:
    d = _profiler_csv_dir()
    if not d.is_dir():
        return {}
    out = {}
    for f in sorted(d.iterdir()):
        m = BATCH_CSV_RE_FILE.match(f.name)
        if m:
            out[m.group(1)] = f
    return out


BATCH_CSVS = discover_batch_csvs()

OOC_NSIGHT_FOCUS_STAGES = [
    'Octree BFS Frustum Culling',
    'Occlusion Culling',
    'Request Generation',
    'Build Active Atom List',
    'Sphere Raster OOC',
]

OOC_PIPELINE_RANGES = [
    'Screen Clear',
    'Octree BFS Frustum Culling',
    'Compute Block Depth+Area',
    'Thrust Sort (Depth)',
    'Occlusion Culling',
    'Request Generation',
    'Build Active Atom List',
    'Sphere Raster OOC',
    'HiZ Downsample',
    'cub::DeviceRadixSort',
    'Blit Framebuffer',
    'Swap Window',
]

FIG_ROOT = Path(r"D:\Users\Escritorio\U\Ot2026-2doSemM\tesis-magister-latex\img\chapter3\results\ooc")
SAVE_FIGURES = True
FIG_DPI = 150

OOC_PIPELINE_EXCLUDED = {
    "Screen Clear", "Blit Framebuffer", "Swap Window",
    "cub::DeviceRadixSort",
}
OOC_PIPELINE_ANALYSIS = [r for r in OOC_PIPELINE_RANGES if r not in OOC_PIPELINE_EXCLUDED]


def _slug(text: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", text).strip("_").lower()


def save_fig(fig, rel_path: str):
    if not SAVE_FIGURES:
        return None
    path = FIG_ROOT / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    return path


def show_and_save(fig, rel_path: str):
    save_fig(fig, rel_path)
    plt.show()


THESIS_PANEL_DPI = 200
PANEL_WIDTH = 14
PANEL_ROW_H = 4.5
PANEL_STALL_ROW_H = 3.5
PANEL_METRIC_ROW_H = 4.0
PANEL_CORR_ROW_H = 5.0


def save_panel_fig(fig, rel_path: str):
    """Save thesis panel at higher DPI."""
    if not SAVE_FIGURES:
        return None
    path = FIG_ROOT / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=THESIS_PANEL_DPI, bbox_inches="tight")
    return path


def plot_metric_by_ring_on_ax(
    ax, df, metric_col, title, ylabel, palette="viridis",
):
    ordered = analysis_stage_order(df["pipeline_range"].values)
    sub = df[df["pipeline_range"].isin(ordered)].copy()
    sub["pipeline_range"] = pd.Categorical(
        sub["pipeline_range"], categories=ordered, ordered=True
    )
    sns.barplot(
        data=sub, x="pipeline_range", y=metric_col, hue="ring",
        palette=palette, ax=ax, errorbar="sd",
    )
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Pipeline stage")
    ax.tick_params(axis="x", rotation=45)
    leg = ax.legend(title="Ring", fontsize=8, title_fontsize=9)
    if leg is not None:
        leg._legend_box.set_alpha(0.9)


def analysis_stage_order(values):
    present = set(values)
    return [r for r in OOC_PIPELINE_ANALYSIS if r in present]


def filter_analysis_df(df):
    return df[df["pipeline_range"].isin(OOC_PIPELINE_ANALYSIS)].copy()


def renormalize_rel_duration(df, group_cols):
    out = df.copy()
    totals = out.groupby(group_cols, observed=True)["rel_frame_duration"].transform("sum")
    out["rel_frame_duration"] = out["rel_frame_duration"] / totals.replace(0, np.nan)
    return out


def prepare_df_for_analysis(df):
    return renormalize_rel_duration(filter_analysis_df(df), ["sample", "ring", "position"])


def prepare_ring_df_for_analysis(df_ring):
    return renormalize_rel_duration(filter_analysis_df(df_ring), ["ring"])

KEY_METRIC_IDS = {
    'gr_active_pct':   'gr__cycles_active.avg.pct_of_peak_sustained_elapsed',
    'gr_idle_pct':     'oracle.gr__cycles_idle.pct',
    'l1tex_hit_pct':   'l1tex__t_sector_hit_rate.pct',
    'pcie_throughput':  'pcie__throughput.avg.pct_of_peak_sustained_elapsed',
    'sm_issue_active':  'sm__inst_executed_realtime.avg.pct_of_peak_sustained_elapsed',
    'sm_pipe_alu':      'sm__inst_executed_pipe_alu_realtime.avg.pct_of_peak_sustained_elapsed',
    'warp_occ_pct':     'sm__warps_active.avg.pct_of_peak_sustained_elapsed',
    'thread_active_pct': 'sm__average_thread_inst_executed_pred_on_per_inst_executed_realtime.pct',
    'stall_short_scoreboard': 'smsp__warps_issue_stalled_short_scoreboard.avg.pct_of_peak_sustained_elapsed',
    'stall_drain':      'smsp__warps_issue_stalled_drain.avg.pct_of_peak_sustained_elapsed',
    'stall_wait':       'smsp__warps_issue_stalled_wait.avg.pct_of_peak_sustained_elapsed',
    'sm_idle_pct':      'tpc__warps_inactive_sm_idle.avg.pct_of_peak_sustained_elapsed',
}

print('Project:', PROJECT_ROOT)
print('CU_OOC root:', CU_OOC_ROOT, '—', 'OK' if CU_OOC_ROOT.is_dir() else 'NOT FOUND')
print(f'Scenes detected: {len(SCENES)}')
for k, v in SCENES.items():
    rings = {int(RANGE_YAML_RE.match(f.name).group(1)) for f in v['yaml_files']}
    poss = {int(RANGE_YAML_RE.match(f.name).group(2)) for f in v['yaml_files']}
    print(f"  [{k}] {v['label']}")
    print(f"       YAML: {v['yaml_dir']}  ({len(v['yaml_files'])} files, rings {min(rings)}–{max(rings)}, positions {sorted(poss)})")
print(f"  Profiler CSV dir: {_profiler_csv_dir()}")
print(f"  Preprocess CSV: {PREPROCESS_CSV} -- {'OK' if PREPROCESS_CSV.exists() else 'NOT FOUND'}")
print(f"  Batch CSVs found: {len(BATCH_CSVS)}")
for bk, bp in BATCH_CSVS.items():
    print(f"    [{bk}] {bp}")

FORCE_OOC_YAML_CACHE_REBUILD = False
OOC_CACHE_DIR = PROJECT_ROOT / 'benchmark' / 'nsight_analysis'
OOC_YAML_CACHE_PKL = OOC_CACHE_DIR / 'ooc_cu_ooc_yaml_agg.pkl'
OOC_YAML_CACHE_META = OOC_CACHE_DIR / 'ooc_cu_ooc_yaml_agg.meta.json'


## 0b. Runtime profiler metrics (`CU_OOC/other_metrics`)

| File | Content |
|------|---------|
| `ooc_stats_batch_<scene>.csv` | Per-batch: **frame time (ms)**, visible/filtered blocks, streaming **requests**, **active atoms** (~15 frames/batch) |
| `ooc_preprocess_metrics.csv` | Preprocess: Morton, blocks file, octree, VRAM |

No `range*_frames.txt` here — use these CSVs for **absolute** timings and culling counters; Nsight YAML gives **GPU stage shares** (`RelativeFrameDuration`).


In [66]:
def load_batch_stats_table():
    if not BATCH_CSVS:
        print("No ooc_stats_batch_*.csv in", _profiler_csv_dir())
        return None
    parts = []
    for bk, bp in BATCH_CSVS.items():
        tmp = pd.read_csv(bp)
        tmp["escena_id"] = bk
        tmp["label"] = SCENE_LABEL_HINTS.get(bk, bk)
        tmp["avg_fps"] = 1000.0 / tmp["avg_frame_ms"].replace(0, np.nan)
        tmp["min_fps"] = 1000.0 / tmp["max_frame_ms"].replace(0, np.nan)
        tmp["max_fps"] = 1000.0 / tmp["min_frame_ms"].replace(0, np.nan)
        tmp["culled_blocks"] = tmp["avg_numVisible"] - tmp["avg_numFiltered"]
        tmp["run_id"] = (tmp["batch_index"].diff() < 0).cumsum()
        tmp["is_full_batch"] = tmp["frames"] == 15
        parts.append(tmp)
        print(f"[OK] batch {bk}: {len(tmp)} rows")
    return pd.concat(parts, ignore_index=True)


def load_preprocess_table():
    if not PREPROCESS_CSV.is_file():
        return None
    df = pd.read_csv(PREPROCESS_CSV)
    df["total_preprocess_ms"] = df["ms_morton_pipeline"] + df["ms_blocks_and_file"] + df["ms_octree_build"]
    print(f"[OK] preprocess: {len(df)} rows")
    return df


df_batch_all = load_batch_stats_table()
df_prep = load_preprocess_table()


[OK] batch 8wql: 4168 rows
[OK] batch g100m: 1732 rows
[OK] batch g10m: 2562 rows
[OK] batch g500m: 2599 rows
[OK] batch g50m: 2401 rows
[OK] preprocess: 47 rows


## 1. YAML parsing (Nsight Trace Analysis)


In [67]:
def _yaml_load(text: str):
    try:
        return yaml.load(text, Loader=yaml.CSafeLoader)
    except AttributeError:
        return yaml.safe_load(text)


def parse_yaml_trace(filepath):
    """Parse a Nsight GPU Trace analysis YAML."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = _yaml_load(f)
    return data if data is not None else []


def _yaml_files_signature(scene_cfg: dict) -> list[dict]:
    sig = []
    for fp in scene_cfg["yaml_files"]:
        st = fp.stat()
        sig.append({"name": fp.name, "mtime": st.st_mtime_ns, "size": st.st_size})
    return sig


def load_or_build_all_scene_data(scenes: dict, cu_ooc_root: Path, force: bool = False) -> dict:
    import json as _json

    root_s = str(cu_ooc_root.resolve())
    meta_ok = False
    if not force and OOC_YAML_CACHE_PKL.is_file() and OOC_YAML_CACHE_META.is_file():
        try:
            meta = _json.loads(OOC_YAML_CACHE_META.read_text(encoding="utf-8"))
            meta_ok = (
                meta.get("nsight_root") == root_s
                and meta.get("pipeline_ranges") == list(OOC_PIPELINE_RANGES)
                and set(meta.get("scene_keys", [])) == set(scenes.keys())
            )
            if meta_ok:
                stored = meta.get("file_sigs", {})
                for sk, cfg in scenes.items():
                    if stored.get(sk) != _yaml_files_signature(cfg):
                        meta_ok = False
                        break
        except Exception:
            meta_ok = False
    if meta_ok:
        out = pd.read_pickle(OOC_YAML_CACHE_PKL)
        print(f"Loaded YAML cache: {len(out)} scenes from {OOC_YAML_CACHE_PKL.name}")
        return out

    print("Building YAML aggregates (first run can take several minutes)…")
    out = {}
    target = set(OOC_PIPELINE_RANGES)
    for scene_key, scene_cfg in scenes.items():
        scene_data = {}
        for fpath in scene_cfg["yaml_files"]:
            m = RANGE_YAML_RE.match(fpath.name)
            if not m:
                continue
            label = f"range{int(m.group(1))}_{int(m.group(2))}"
            try:
                scene_data[label] = aggregate_yaml_file(str(fpath), target)
            except Exception as e:
                print(f"[ERROR] {scene_key} / {fpath.name}: {e}")
        out[scene_key] = scene_data
        print(f"[OK] {scene_key}: {len(scene_data)} YAML samples parsed")

    OOC_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(out, OOC_YAML_CACHE_PKL)
    OOC_YAML_CACHE_META.write_text(
        _json.dumps(
            {
                "nsight_root": root_s,
                "pipeline_ranges": list(OOC_PIPELINE_RANGES),
                "scene_keys": sorted(scenes.keys()),
                "file_sigs": {sk: _yaml_files_signature(scenes[sk]) for sk in scenes},
                "n_scenes": len(out),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Wrote YAML cache -> {OOC_YAML_CACHE_PKL}")
    return out


def aggregate_metrics_by_ring(df: pd.DataFrame) -> pd.DataFrame:
    """Mean over YAML replicas (position 1..3) per camera ring and pipeline stage."""
    metric_cols = [
        c
        for c in df.columns
        if c not in ("sample", "ring", "position", "pipeline_range", "rel_frame_duration")
    ]
    agg = {c: "mean" for c in metric_cols}
    agg["rel_frame_duration"] = "mean"
    agg["sample"] = "count"
    return (
        df.groupby(["ring", "pipeline_range"], observed=True)
        .agg(agg)
        .reset_index()
        .rename(columns={"sample": "n_yaml_replicas"})
    )


def extract_metrics_from_range_entry(entry):
    """From one YAML range entry, extract all metric Id→Value pairs."""
    metrics = {}
    for rule in entry.get('Rules', []) or []:
        for m in rule.get('Metrics', []) or []:
            mid = m.get('Id', '')
            val = m.get('Value')
            if mid and val is not None:
                try:
                    metrics[mid] = float(val)
                except (ValueError, TypeError):
                    pass
    metrics['RelativeFrameDuration'] = entry.get('RelativeFrameDuration', 0.0)
    return metrics


def aggregate_yaml_file(filepath, target_ranges=None):
    """
    Parse one YAML file with multiple frames of Nsight data.
    Each range name repeats once per frame. We average metrics across frames
    for each unique range name.
    Returns dict: range_name -> {metric_id: avg_value}
    """
    entries = parse_yaml_trace(filepath)
    buckets = defaultdict(list)  # range_name -> [metrics_dict, ...]
    for e in entries:
        rname = e.get('Range', '')
        if target_ranges and rname not in target_ranges:
            continue
        buckets[rname].append(extract_metrics_from_range_entry(e))

    result = {}
    for rname, mlist in buckets.items():
        all_keys = set()
        for m in mlist:
            all_keys.update(m.keys())
        avg = {}
        for k in all_keys:
            vals = [m[k] for m in mlist if k in m]
            if vals:
                avg[k] = np.mean(vals)
        result[rname] = avg
    return result


print('Parsing helpers ready.')


Parsing helpers ready.


## 2. Load all YAML files per scene


In [68]:
all_scene_data = load_or_build_all_scene_data(
    SCENES, CU_OOC_ROOT, force=FORCE_OOC_YAML_CACHE_REBUILD
)

if not all_scene_data:
    print("No data loaded. Add range*_analysis.yaml under", CU_OOC_ROOT)


Loaded YAML cache: 5 scenes from ooc_cu_ooc_yaml_agg.pkl


### Notes on CU_OOC metrics

- **`ring`** = camera distance (`range1`…`range5` in the filename); **`position`** = YAML replica (1–3).
- **`RelativeFrameDuration`** is the fraction of frame time per pipeline stage.
- **Analysis stages** exclude `Screen Clear`, `Blit Framebuffer`, `Swap Window`, and `cub::DeviceRadixSort` (eight core stages); remaining fractions are **renormalized** to sum to 1 per capture.
- There are **no** `range*_frames.txt` under CU_OOC — only relative stage shares and Nsight counters, not absolute ms per stage.
- **`HiZ Downsample`** often has no GPU counters (NaN); duration share is still valid.
- Bar charts in §6–8 use **`dfs_ring`**. §9 keeps **`dfs`** for error bars across replicas.
- With **`SAVE_FIGURES = True`**, plots are saved under `img/chapter3/results/ooc/` in the thesis tree.


### 2b. Extraction summary table (all CU\_OOC scenes)


In [69]:
rows_sum = []
for sk, sd in all_scene_data.items():
    cfg = SCENES[sk]
    rings = [int(RANGE_YAML_RE.match(f.name).group(1)) for f in cfg['yaml_files']]
    poss = [int(RANGE_YAML_RE.match(f.name).group(2)) for f in cfg['yaml_files']]
    rows_sum.append({
        'escena_id': sk,
        'label': cfg['label'],
        'n_yaml': len(cfg['yaml_files']),
        'ring_min': min(rings) if rings else None,
        'ring_max': max(rings) if rings else None,
        'positions': sorted(set(poss)),
        'yaml_folder': str(cfg['yaml_dir']),
    })
df_extract = pd.DataFrame(rows_sum)
if not df_extract.empty:
    display(df_extract)
else:
    print('No rows: no scenes with valid YAML.')


,escena_id,label,n_yaml,ring_min,ring_max,positions,yaml_folder
0,g10m,PACKAGE_SCENE ~10M spheres,15,1,5,"[1, 2, 3]",d:\Users\Escritorio\Rasterization-Software\ben...
1,g50m,PACKAGE_SCENE ~50M spheres,15,1,5,"[1, 2, 3]",d:\Users\Escritorio\Rasterization-Software\ben...
2,g100m,PACKAGE_SCENE ~100M spheres,15,1,5,"[1, 2, 3]",d:\Users\Escritorio\Rasterization-Software\ben...
3,g500m,PACKAGE_SCENE ~500M spheres,15,1,5,"[1, 2, 3]",d:\Users\Escritorio\Rasterization-Software\ben...
4,8wql,Molecule 8wql (LOADED_SCENE),15,1,5,"[1, 2, 3]",d:\Users\Escritorio\Rasterization-Software\ben...


## 3. Consolidated GPU metrics DataFrame per pipeline range


In [ ]:
def build_metrics_dataframe(scene_data, key_metrics=KEY_METRIC_IDS):
    rows = []
    for sample_label, ranges in scene_data.items():
        parts = sample_label.replace('range', '').split('_')
        ring, pos = int(parts[0]), int(parts[1])
        for rname, mdict in ranges.items():
            row = {
                'sample': sample_label, 'ring': ring, 'position': pos,
                'pipeline_range': rname,
                'rel_frame_duration': mdict.get('RelativeFrameDuration', 0.0),
            }
            for friendly, mid in key_metrics.items():
                row[friendly] = mdict.get(mid, np.nan)
            rows.append(row)
    return pd.DataFrame(rows)


dfs = {}
dfs_raw = {}
for scene_key, scene_data in all_scene_data.items():
    df_raw = build_metrics_dataframe(scene_data)
    dfs_raw[scene_key] = df_raw
    df = prepare_df_for_analysis(df_raw)
    dfs[scene_key] = df
    print(f"\n=== {SCENES[scene_key]['label']} ===")
    print(f"Rows: {len(df)}, unique samples: {df['sample'].nunique()}, ranges: {df['pipeline_range'].nunique()}")
    display(df.head(16))

dfs_ring = {sk: prepare_ring_df_for_analysis(aggregate_metrics_by_ring(dfs_raw[sk])) for sk in dfs_raw}
print("Per-ring aggregates (mean over 3 YAML replicas per camera range):")
for sk, dr in dfs_ring.items():
    print(f"  {sk}: {len(dr)} rows")


### 3c. Nsight + runtime profiler (per scene)

Join mean Nsight **stage shares** (`dfs_ring`) with mean **frame ms / culling / requests** from `other_metrics`. Does not map each `batch_index` to each `rangeN_pos` (continuous log vs isolated captures).


In [ ]:
def _rel_col(stage):
    return "rel_" + stage.replace(" ", "_").replace("+", "plus")


def nsight_scene_summary(dfs_ring_map):
    rows = []
    for sk, dr in dfs_ring_map.items():
        row = {"escena_id": sk}
        for stage in OOC_NSIGHT_FOCUS_STAGES:
            row[_rel_col(stage)] = dr.loc[dr["pipeline_range"] == stage, "rel_frame_duration"].mean()
        row["rel_ooc_core"] = dr.loc[dr["pipeline_range"].isin(OOC_NSIGHT_FOCUS_STAGES), "rel_frame_duration"].sum()
        rows.append(row)
    return pd.DataFrame(rows)


if df_batch_all is not None and not df_batch_all.empty and "dfs_ring" in dir():
    prof = (
        df_batch_all.groupby("escena_id", observed=True)
        .agg(
            frame_ms_mean=("avg_frame_ms", "mean"), fps_mean=("avg_fps", "mean"),
            visible_mean=("avg_numVisible", "mean"), filtered_mean=("avg_numFiltered", "mean"),
            requests_mean=("avg_numRequests", "mean"), active_mean=("avg_activeCount", "mean"),
        ).reset_index()
    )
    joined = nsight_scene_summary(dfs_ring).merge(prof, on="escena_id")
    display(joined.round(4))
    pkg = joined[joined["escena_id"].str.match(r"^g\d+m$", na=False)].copy()
    if len(pkg) >= 2:
        pkg["sphere_M"] = pkg["escena_id"].str.extract(r"^g(\d+)m$")[0].astype(float)
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].scatter(pkg["sphere_M"], pkg["frame_ms_mean"], s=90)
        ax[0].set_xscale("log")
        ax[0].set_xlabel("Spheres (millions)")
        ax[0].set_ylabel("Mean frame time (ms)")
        ax[1].scatter(pkg[_rel_col("Build Active Atom List")], pkg["active_mean"], s=90)
        ax[1].set_xlabel("Nsight rel. Build Active Atom List")
        ax[1].set_ylabel("Mean active atoms / batch")
        plt.tight_layout()
        show_and_save(fig, "package/nsight_vs_profiler_frame_ms.png")


### 3b. Overview: `g*m` scenes vs inferred size (millions of spheres)

If folders follow the `g10m`, `g50m`, etc. pattern, the number is interpreted as millions of entities for **comparative plots** across scenes only (does not replace `sphere_count` from the preprocess CSV).


In [ ]:
def infer_sphere_millions(scene_id: str):
    m = re.match(r'^g(\d+)m$', scene_id, re.IGNORECASE)
    return int(m.group(1)) if m else None


pkg_scenes = [(k, infer_sphere_millions(k)) for k in dfs if infer_sphere_millions(k) is not None]
if len(pkg_scenes) >= 2:
    target_rng = 'Sphere Raster OOC'
    all_rings = set()
    for sk, _ in pkg_scenes:
        all_rings.update(dfs[sk]['ring'].unique())
    fig, ax = plt.subplots(figsize=(11, 5))
    for ring_id in sorted(all_rings):
        xs, ys = [], []
        for sk, millions in sorted(pkg_scenes, key=lambda t: t[1]):
            sub = dfs[sk][(dfs[sk]['pipeline_range'] == target_rng) & (dfs[sk]['ring'] == ring_id)]
            if sub.empty:
                continue
            xs.append(millions)
            ys.append(sub['rel_frame_duration'].mean())
        if len(xs) >= 2:
            ax.plot(xs, ys, marker='o', label=f'Ring {ring_id}')
    if ax.lines:
        ax.set_xscale('log')
        ax.set_xlabel('Spheres (millions, log scale)')
        ax.set_ylabel(f'Mean frame fraction — {target_rng}')
        ax.set_title('PACKAGE scales: relative OOC raster cost vs scene size')
        ax.legend(title='Distance (ring)')
        plt.tight_layout()
        show_and_save(fig, "package/raster_fraction_vs_scale.png")
    else:
        print('Not enough points for', target_rng, 'in g*m scenes.')
else:
    print('Fewer than 2 g*m folders with data; skipping comparative overview.')


## 4. Relative duration per pipeline stage (frame fraction)


In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    pivot = df.pivot_table(index='pipeline_range', columns='ring', values='rel_frame_duration', aggfunc='mean')
    ordered = analysis_stage_order(pivot.index)
    pivot = pivot.loc[ordered]

    fig, ax = plt.subplots(figsize=(10, 6))
    pivot.T.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
    ax.set_title(f"{cfg['label']} — Frame fraction by stage")
    ax.set_xlabel('Ring (distance)')
    ax.set_ylabel('Frame fraction')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.set_xticklabels([f'Ring {c}' for c in pivot.columns], rotation=0)
    plt.tight_layout()
    show_and_save(fig, f"nsight/stage_fraction/{scene_key}.png")

    fig, ax = plt.subplots(figsize=(12, max(6, len(ordered) * 0.45)))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Frame fraction'})
    ax.set_title(f"{cfg['label']} — Relative duration heatmap")
    ax.set_ylabel('Pipeline stage')
    ax.set_xlabel('Ring')
    plt.tight_layout()
    show_and_save(fig, f"nsight/stage_heatmap/{scene_key}.png")


## 5. Bottleneck: most expensive stage per ring


In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    grp = df.groupby(['ring', 'pipeline_range'])['rel_frame_duration'].mean().reset_index()
    idx = grp.groupby('ring')['rel_frame_duration'].idxmax()
    bottleneck = grp.loc[idx][['ring', 'pipeline_range', 'rel_frame_duration']]
    bottleneck.columns = ['Ring', 'Bottleneck stage', 'Frame fraction']
    print(f"\n=== {cfg['label']} — Bottleneck by distance ===")
    display(bottleneck.reset_index(drop=True))


## 6. GR Cycles Active & GPU Idle by stage and ring


In [ ]:
def plot_metric_by_ring(df, metric_col, title, ylabel, rel_path,
                        palette='viridis', figsize=(16, 7)):
    fig, ax = plt.subplots(figsize=figsize)
    plot_metric_by_ring_on_ax(
        ax, df, metric_col, title, ylabel, palette=palette,
    )
    plt.tight_layout()
    show_and_save(fig, rel_path)


for scene_key, df in dfs_ring.items():
    cfg = SCENES[scene_key]
    plot_metric_by_ring(df, 'gr_active_pct', f"{cfg['label']} — GR Cycles Active [%]", 'GR Active %',
                        f"nsight/gr_active/{scene_key}.png")
    plot_metric_by_ring(df, 'gr_idle_pct', f"{cfg['label']} — GPU Idle [%]", 'GR Idle %',
                        f"nsight/gr_idle/{scene_key}.png")


## 7. L1TEX Hit Rate and PCIe Throughput


In [ ]:
for scene_key, df in dfs_ring.items():
    cfg = SCENES[scene_key]
    plot_metric_by_ring(df, 'l1tex_hit_pct', f"{cfg['label']} — L1TEX Hit Rate [%]", 'L1TEX Hit %',
                        f"nsight/l1tex_hit/{scene_key}.png")
    plot_metric_by_ring(df, 'pcie_throughput', f"{cfg['label']} — PCIe Throughput [%]", 'PCIe %',
                        f"nsight/pcie/{scene_key}.png", palette='magma')


## 8. Warp Occupancy and Stall Breakdown


In [ ]:
stall_cols = ['stall_short_scoreboard', 'stall_drain', 'stall_wait']

for scene_key, df in dfs_ring.items():
    cfg = SCENES[scene_key]
    plot_metric_by_ring(df, 'warp_occ_pct', f"{cfg['label']} — SM Warp Occupancy [%]", 'Warp Occupancy %',
                        f"nsight/warp_occupancy/{scene_key}.png", palette='crest')
    stall_data = df.groupby(['ring', 'pipeline_range'])[stall_cols].mean().reset_index()
    ordered = analysis_stage_order(stall_data['pipeline_range'].values)
    for ring_id in sorted(df['ring'].unique()):
        sub = stall_data[(stall_data['ring'] == ring_id) & stall_data['pipeline_range'].isin(ordered)].copy()
        sub['pipeline_range'] = pd.Categorical(sub['pipeline_range'], categories=ordered, ordered=True)
        sub = sub.sort_values('pipeline_range').set_index('pipeline_range')[stall_cols]
        if sub.dropna(how='all').empty:
            continue
        fig, ax = plt.subplots(figsize=(14, 5))
        sub.plot(kind='bar', stacked=True, ax=ax, color=['#e74c3c', '#3498db', '#2ecc71'])
        ax.set_title(f"{cfg['label']} — Stall Breakdown, Ring {ring_id}")
        ax.set_ylabel('Stall %')
        ax.set_xlabel('Pipeline stage')
        ax.tick_params(axis='x', rotation=45)
        ax.legend(['Short Scoreboard', 'Drain', 'Wait'])
        plt.tight_layout()
        show_and_save(fig, f"nsight/stall_breakdown/{scene_key}_ring{ring_id}.png")


## 9. Key metrics vs distance (trend by ring)

For the most relevant OOC pipeline stages (Sphere Raster, Frustum Culling, Occlusion Culling), we plot how metrics change as the camera moves away.


In [ ]:
focus_ranges = ['Sphere Raster OOC', 'Octree BFS Frustum Culling',
                'Occlusion Culling', 'Build Active Atom List']
focus_metrics = ['gr_active_pct', 'gr_idle_pct', 'l1tex_hit_pct',
                 'pcie_throughput', 'warp_occ_pct', 'sm_issue_active']

for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    sub = df[df['pipeline_range'].isin(focus_ranges)].copy()
    if sub.empty:
        continue
    for met in focus_metrics:
        if sub[met].dropna().empty:
            continue
        fig, ax = plt.subplots(figsize=(12, 5))
        for rng in focus_ranges:
            rng_data = sub[sub['pipeline_range'] == rng].groupby('ring')[met].agg(['mean', 'std']).reset_index()
            if rng_data['mean'].dropna().empty:
                continue
            ax.errorbar(rng_data['ring'], rng_data['mean'], yerr=rng_data['std'],
                        marker='o', capsize=4, label=rng)
        ax.set_title(f"{cfg['label']} — {met} vs distance")
        ax.set_xlabel('Ring (1=near, 5=far)')
        ax.set_ylabel(met)
        ax.legend(fontsize=8)
        ax.set_xticks(sorted(df['ring'].unique()))
        plt.tight_layout()
        show_and_save(fig, f"nsight/metrics_vs_ring/{scene_key}_{_slug(met)}.png")


## 10. Metric correlation heatmap (main stages)


In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    for rng in focus_ranges:
        sub = df[df['pipeline_range'] == rng]
        cols = [c for c in focus_metrics if not sub[c].dropna().empty]
        if len(cols) < 2:
            continue
        corr = sub[cols].corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
        ax.set_title(f"{cfg['label']} — Metric correlation: {rng}")
        plt.tight_layout()
        show_and_save(fig, f"nsight/correlation/{scene_key}_{_slug(rng)}.png")


## 10b. Thesis panels — vertical stacks for LaTeX (full-width figures)


In [ ]:
focus_ranges = ['Sphere Raster OOC', 'Octree BFS Frustum Culling',
                'Occlusion Culling', 'Build Active Atom List']
focus_metrics = ['gr_active_pct', 'gr_idle_pct', 'l1tex_hit_pct',
                 'pcie_throughput', 'warp_occ_pct', 'sm_issue_active']
stall_cols = ['stall_short_scoreboard', 'stall_drain', 'stall_wait']


def _export_gr_active_idle_panel(scene_key='g500m'):
    df = dfs_ring[scene_key]
    cfg = SCENES[scene_key]
    n = 2
    fig, axes = plt.subplots(n, 1, figsize=(PANEL_WIDTH, PANEL_ROW_H * n), squeeze=False)
    plot_metric_by_ring_on_ax(
        axes[0, 0], df, 'gr_active_pct',
        f"{cfg['label']} — GR Cycles Active [%]", 'GR Active %',
    )
    plot_metric_by_ring_on_ax(
        axes[1, 0], df, 'gr_idle_pct',
        f"{cfg['label']} — GPU Idle [%]", 'GR Idle %',
    )
    fig.tight_layout()
    save_panel_fig(fig, f"nsight/panels/gr_active_idle_{scene_key}.png")
    plt.close(fig)


def _export_l1tex_pcie_panel(scene_key='g500m'):
    df = dfs_ring[scene_key]
    cfg = SCENES[scene_key]
    n = 2
    fig, axes = plt.subplots(n, 1, figsize=(PANEL_WIDTH, PANEL_ROW_H * n), squeeze=False)
    plot_metric_by_ring_on_ax(
        axes[0, 0], df, 'l1tex_hit_pct',
        f"{cfg['label']} — L1TEX Hit Rate [%]", 'L1TEX Hit %',
    )
    plot_metric_by_ring_on_ax(
        axes[1, 0], df, 'pcie_throughput',
        f"{cfg['label']} — PCIe Throughput [%]", 'PCIe %', palette='magma',
    )
    fig.tight_layout()
    save_panel_fig(fig, f"nsight/panels/l1tex_pcie_{scene_key}.png")
    plt.close(fig)


def _export_stall_rings_panel(scene_key):
    df = dfs_ring[scene_key]
    cfg = SCENES[scene_key]
    rings = sorted(df['ring'].unique())
    stall_data = df.groupby(['ring', 'pipeline_range'])[stall_cols].mean().reset_index()
    ordered = analysis_stage_order(stall_data['pipeline_range'].values)
    n = len(rings)
    fig, axes = plt.subplots(n, 1, figsize=(PANEL_WIDTH, PANEL_STALL_ROW_H * n), squeeze=False)
    for i, ring_id in enumerate(rings):
        ax = axes[i, 0]
        sub = stall_data[(stall_data['ring'] == ring_id) & stall_data['pipeline_range'].isin(ordered)].copy()
        sub['pipeline_range'] = pd.Categorical(sub['pipeline_range'], categories=ordered, ordered=True)
        sub = sub.sort_values('pipeline_range').set_index('pipeline_range')[stall_cols]
        if sub.dropna(how='all').empty:
            ax.set_visible(False)
            continue
        sub.plot(kind='bar', stacked=True, ax=ax, color=['#e74c3c', '#3498db', '#2ecc71'], legend=False)
        ax.set_title(f"Ring {ring_id}", fontsize=11)
        ax.set_ylabel('Stall %')
        ax.set_xlabel('Pipeline stage')
        ax.tick_params(axis='x', rotation=45)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if not handles:
        from matplotlib.patches import Patch
        handles = [Patch(facecolor=c) for c in ['#e74c3c', '#3498db', '#2ecc71']]
        labels = ['Short Scoreboard', 'Drain', 'Wait']
    fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.98, 0.98), fontsize=9)
    fig.suptitle(f"{cfg['label']} — Stall breakdown by ring", fontsize=12, y=1.002)
    fig.tight_layout()
    save_panel_fig(fig, f"nsight/panels/stall_rings_{scene_key}.png")
    plt.close(fig)


def _export_metrics_vs_ring_panel(scene_key):
    df = dfs[scene_key]
    cfg = SCENES[scene_key]
    sub = df[df['pipeline_range'].isin(focus_ranges)].copy()
    metrics = [m for m in focus_metrics if not sub[m].dropna().empty]
    if not metrics:
        return
    n = len(metrics)
    fig, axes = plt.subplots(n, 1, figsize=(PANEL_WIDTH, PANEL_METRIC_ROW_H * n), squeeze=False)
    rings = sorted(df['ring'].unique())
    for i, met in enumerate(metrics):
        ax = axes[i, 0]
        for rng in focus_ranges:
            rng_data = sub[sub['pipeline_range'] == rng].groupby('ring')[met].agg(['mean', 'std']).reset_index()
            if rng_data['mean'].dropna().empty:
                continue
            ax.errorbar(rng_data['ring'], rng_data['mean'], yerr=rng_data['std'].fillna(0),
                        marker='o', capsize=4, label=rng)
        ax.set_title(f"{met}", fontsize=11)
        ax.set_xlabel('Ring (1=near, 5=far)')
        ax.set_ylabel(met)
        ax.legend(fontsize=7, loc='best')
        ax.set_xticks(rings)
    fig.suptitle(f"{cfg['label']} — Metrics vs camera ring", fontsize=12, y=1.002)
    fig.tight_layout()
    save_panel_fig(fig, f"nsight/panels/metrics_vs_ring_{scene_key}.png")
    plt.close(fig)


def _export_correlation_panel(scene_key='g10m'):
    df = dfs[scene_key]
    cfg = SCENES[scene_key]
    ranges = [r for r in focus_ranges if len(df[df['pipeline_range'] == r]) > 0]
    n = len(ranges)
    fig, axes = plt.subplots(n, 1, figsize=(PANEL_WIDTH, PANEL_CORR_ROW_H * n), squeeze=False)
    for i, rng in enumerate(ranges):
        sub = df[df['pipeline_range'] == rng]
        cols = [c for c in focus_metrics if not sub[c].dropna().empty]
        if len(cols) < 2:
            axes[i, 0].set_visible(False)
            continue
        corr = sub[cols].corr()
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
                    ax=axes[i, 0], cbar_kws={'shrink': 0.8})
        axes[i, 0].set_title(f"{rng}", fontsize=11)
    fig.suptitle(f"{cfg['label']} — Metric correlation by stage", fontsize=12, y=1.002)
    fig.tight_layout()
    save_panel_fig(fig, f"nsight/panels/correlation_{scene_key}.png")
    plt.close(fig)


if 'dfs_ring' in dir() and 'g500m' in dfs_ring:
    _export_gr_active_idle_panel('g500m')
    _export_l1tex_pcie_panel('g500m')
if 'dfs_ring' in dir():
    for sk in ('g100m', '8wql'):
        if sk in dfs_ring:
            _export_stall_rings_panel(sk)
if 'dfs' in dir():
    for sk in ('g10m', 'g500m'):
        if sk in dfs:
            _export_metrics_vs_ring_panel(sk)
    if 'g10m' in dfs:
        _export_correlation_panel('g10m')

print('Thesis panels exported under', FIG_ROOT / 'nsight' / 'panels')


## 11. OOC preprocess — timing, scaling, and memory

Analysis of `ooc_preprocess_metrics.csv`. Columns: Morton, blocks/file, octree timings; host/block-file sizes; estimated VRAM.


In [ ]:
if df_prep is None and PREPROCESS_CSV.is_file():
    df_prep = load_preprocess_table()

if df_prep is not None:
    if 'atoms_per_ms' not in df_prep.columns:
        df_prep['total_preprocess_ms'] = df_prep[['ms_morton_pipeline','ms_blocks_and_file','ms_octree_build']].sum(axis=1)
        df_prep['atoms_per_ms'] = df_prep['sphere_count'] / df_prep['total_preprocess_ms']
        df_prep['vram_used_MB'] = df_prep['vram_sum_estimated_bytes'] / (1024**2)
        df_prep['block_file_MB'] = df_prep['block_file_bytes'] / (1024**2)
        df_prep['host_MB'] = df_prep['host_structures_bytes'] / (1024**2)
        df_prep['vram_pct_used'] = 100.0 * (1.0 - df_prep['cuda_mem_free_bytes'] / df_prep['cuda_mem_total_bytes'])

    prep_grp = df_prep.groupby(['scene_type','sphere_count']).agg(
        n_runs=('total_preprocess_ms','count'),
        avg_ms=('total_preprocess_ms','mean'), std_ms=('total_preprocess_ms','std'),
        min_ms=('total_preprocess_ms','min'), max_ms=('total_preprocess_ms','max'),
        avg_morton=('ms_morton_pipeline','mean'), avg_blocks=('ms_blocks_and_file','mean'),
        avg_octree=('ms_octree_build','mean'), avg_throughput=('atoms_per_ms','mean'),
        avg_vram_MB=('vram_used_MB','mean'), avg_vram_pct=('vram_pct_used','mean'),
        avg_block_file_MB=('block_file_MB','mean'), avg_host_MB=('host_MB','mean'),
        block_count=('block_count','first'), octree_nodes=('octree_node_count','first'),
    ).reset_index().sort_values('sphere_count')

    print('=== Preprocess: summary by scene/size ===')
    display(prep_grp)

    pkg = prep_grp[prep_grp['scene_type'].str.contains('PACKAGE')].copy()
    if len(pkg) >= 2:
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        ax = axes[0,0]
        ax.errorbar(pkg['sphere_count']/1e6, pkg['avg_ms'], yerr=pkg['std_ms'].fillna(0), marker='o', capsize=4)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('Total time (ms)')
        ax.set_title('Preprocess time vs entity count')
        ax = axes[0,1]
        for col, lbl in [('avg_morton','Morton+sort'), ('avg_blocks','Blocks+file'), ('avg_octree','Octree')]:
            ax.plot(pkg['sphere_count']/1e6, pkg[col], marker='o', label=lbl)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('ms')
        ax.set_title('Preprocess phase breakdown'); ax.legend()
        ax = axes[0,2]
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_throughput'], marker='s', color='green')
        ax.set_xscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('Atoms/ms')
        ax.set_title('Preprocess throughput')
        ax = axes[1,0]
        ax.bar(pkg['sphere_count'].astype(str), pkg['avg_vram_MB'], color='steelblue')
        ax.set_xlabel('Spheres'); ax.set_ylabel('Estimated VRAM (MB)')
        ax.set_title('Total estimated VRAM'); ax.tick_params(axis='x', rotation=45)
        ax = axes[1,1]
        ax.bar(pkg['sphere_count'].astype(str), pkg['avg_vram_pct'], color='coral')
        ax.set_xlabel('Spheres'); ax.set_ylabel('% VRAM used')
        ax.set_title('% GPU VRAM used'); ax.tick_params(axis='x', rotation=45)
        ax = axes[1,2]
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_block_file_MB'], marker='o', label='Block file')
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_host_MB'], marker='^', label='Host structs')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('MB')
        ax.set_title('On-disk and host data'); ax.legend()
        plt.tight_layout()
        show_and_save(fig, "preprocess/preprocess_overview.png")
else:
    print(f'\u26a0 Preprocess CSV not found at {PREPROCESS_CSV}')


## 12. Batch stats — per-scene runtime analysis

Each `ooc_stats_batch_<scene>.csv` file contains rows aggregated over N frames.

**Significado de las columnas clave:**
- `avg_numVisible`: mean **blocks** visible after octree frustum culling.
- `avg_numFiltered`: mean **blocks** that pass occlusion culling (subset of visible blocks that are actually processed).
- `avg_numRequests`: mean **blocks** newly requested for streaming (not yet resident in the GPU pool).
- `avg_activeCount`: mean **atoms actually rasterized** by the `Sphere Raster OOC` kernel.


In [ ]:
if df_batch_all is None:
    df_batch_all = load_batch_stats_table()

if df_batch_all is None or df_batch_all.empty:
    print('No batch stats in', _profiler_csv_dir())
else:
    print('\n=== Summary by scene ===')
    summary = df_batch_all.groupby('escena_id').agg(
        sphere_count=('sphere_count','first'), total_blocks=('total_blocks','first'),
        pool_slots=('pool_slots','first'), n_batches=('avg_fps','count'),
        fps_mean=('avg_fps','mean'), fps_std=('avg_fps','std'),
        frame_ms_mean=('avg_frame_ms','mean'), visible_mean=('avg_numVisible','mean'),
        filtered_mean=('avg_numFiltered','mean'), requests_mean=('avg_numRequests','mean'),
        active_mean=('avg_activeCount','mean'),
    ).reset_index()
    display(summary)

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_fps', order=order, ax=axes[0,0], palette='viridis')
    axes[0,0].set_title('FPS distribution by scene'); axes[0,0].set_xlabel('Scene'); axes[0,0].set_ylabel('FPS')
    axes[0,0].tick_params(axis='x', rotation=30)
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_frame_ms', order=order, ax=axes[0,1], palette='magma')
    axes[0,1].set_title('Frame time (ms) by scene'); axes[0,1].set_xlabel('Scene'); axes[0,1].set_ylabel('ms')
    axes[0,1].tick_params(axis='x', rotation=30)
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_activeCount', order=order, ax=axes[1,0], palette='crest')
    axes[1,0].set_title('Rasterized atoms by scene'); axes[1,0].set_xlabel('Scene'); axes[1,0].set_ylabel('Atoms')
    axes[1,0].tick_params(axis='x', rotation=30)
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_numRequests', order=order, ax=axes[1,1], palette='flare')
    axes[1,1].set_title('Streaming-requested blocks by scene'); axes[1,1].set_xlabel('Scene')
    axes[1,1].set_ylabel('Requests/batch'); axes[1,1].tick_params(axis='x', rotation=30)
    plt.tight_layout()
    show_and_save(fig, "batch/summary_boxplots.png")

    for bk in order:
        sub = df_batch_all[df_batch_all['escena_id'] == bk].copy()
        lbl = sub['label'].iloc[0]
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f'{lbl} ({bk})', fontsize=14)
        axes[0,0].plot(sub['batch_index'], sub['avg_fps'], marker='.', ms=3)
        axes[0,0].fill_between(sub['batch_index'], sub['min_fps'], sub['max_fps'], alpha=0.15)
        axes[0,0].set_title('FPS'); axes[0,0].set_xlabel('Batch'); axes[0,0].set_ylabel('FPS')
        axes[0,1].plot(sub['batch_index'], sub['avg_numVisible'], label='Visible (frustum)', ms=3, marker='.')
        axes[0,1].plot(sub['batch_index'], sub['avg_numFiltered'], label='Post-occlusion', ms=3, marker='.')
        axes[0,1].set_title('Blocks: visible vs post-occlusion'); axes[0,1].legend()
        axes[1,0].plot(sub['batch_index'], sub['avg_activeCount'], color='green', ms=3, marker='.')
        axes[1,0].fill_between(sub['batch_index'], sub['min_activeCount'], sub['max_activeCount'], alpha=0.15, color='green')
        axes[1,0].set_title('Rasterized atoms'); axes[1,0].set_ylabel('Atoms')
        axes[1,1].plot(sub['batch_index'], sub['avg_numRequests'], color='orange', ms=3, marker='.')
        axes[1,1].fill_between(sub['batch_index'], sub['min_numRequests'], sub['max_numRequests'], alpha=0.15, color='orange')
        axes[1,1].set_title('Blocks requested for streaming'); axes[1,1].set_ylabel('Requests')
        plt.tight_layout()
        show_and_save(fig, f"batch/timeseries/{bk}.png")

    fig, ax = plt.subplots(figsize=(12, 7))
    for bk in order:
        sub = df_batch_all[df_batch_all['escena_id'] == bk]
        ax.scatter(sub['avg_activeCount'], sub['avg_fps'], s=15, alpha=0.5, label=bk)
    ax.set_xlabel('Mean rasterized atoms'); ax.set_ylabel('Mean FPS')
    ax.set_title('FPS vs rasterized atoms (all scenes)')
    ax.legend(title='Scene')
    plt.tight_layout()
    show_and_save(fig, "batch/fps_vs_active_atoms.png")


### 12b. Frame time vs batch counters (per scene)

Batch CSV rows are aggregated over camera motion and do **not** carry ring labels.
Each point is one batch; scatter plots relate mean frame time to the four logical counters
within each scene separately.


In [ ]:
FRAME_MS_COUNTERS = [
    ('avg_numRequests', 'Streaming requests (blocks/batch)'),
    ('avg_numFiltered', 'Post-occlusion blocks (blocks/batch)'),
    ('avg_numVisible', 'Frustum-visible blocks (blocks/batch)'),
    ('avg_activeCount', 'Rasterized atoms (activeCount)'),
]

if df_batch_all is not None and not df_batch_all.empty:
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    for bk in order:
        sub = df_batch_all[df_batch_all['escena_id'] == bk].copy()
        lbl = sub['label'].iloc[0]
        fig, axes = plt.subplots(2, 2, figsize=(14, 11))
        fig.suptitle(f'{lbl} — Mean frame time vs batch counters', fontsize=13)
        for ax, (col, ylabel) in zip(axes.flat, FRAME_MS_COUNTERS):
            valid = sub[[col, 'avg_frame_ms']].dropna()
            ax.scatter(valid[col], valid['avg_frame_ms'], s=28, alpha=0.55, edgecolors='none')
            ax.set_xlabel(ylabel)
            ax.set_ylabel('Mean frame time (ms)')
        plt.tight_layout()
        show_and_save(fig, f"batch/frame_ms_scatter/{bk}.png")
    print('Frame-ms scatter panels exported under', FIG_ROOT / 'batch' / 'frame_ms_scatter')


### 12f. Scaling: mean FPS vs sphere count (PACKAGE_SCENE)

Log-log curve of mean FPS per scene. Ideal slope: -1 (linear). Includes a trend line.


In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    pkg_only = df_batch_all[df_batch_all['scene_type'].str.contains('PACKAGE', na=False)].copy()
    if not pkg_only.empty:
        sc_grp = pkg_only.groupby('escena_id').agg(
            sphere_count=('sphere_count','first'),
            fps_mean=('avg_fps','mean'), fps_std=('avg_fps','std'),
            fps_median=('avg_fps', 'median'),
            fps_p25=('avg_fps', lambda x: x.quantile(0.25)),
            fps_p75=('avg_fps', lambda x: x.quantile(0.75)),
        ).reset_index().sort_values('sphere_count')

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        ax = axes[0]
        ax.errorbar(sc_grp['sphere_count']/1e6, sc_grp['fps_mean'], yerr=sc_grp['fps_std'].fillna(0),
                    marker='o', capsize=5, lw=2)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('Mean FPS')
        ax.set_title('FPS vs entity count'); ax.grid(True, which='both', alpha=0.3)
        if len(sc_grp) >= 2:
            coeffs = np.polyfit(np.log10(sc_grp['sphere_count']), np.log10(sc_grp['fps_mean']), 1)
            xs = np.logspace(np.log10(sc_grp['sphere_count'].min()), np.log10(sc_grp['sphere_count'].max()), 50)
            ax.plot(xs/1e6, 10**(coeffs[0]*np.log10(xs) + coeffs[1]), '--', color='red',
                    label=f'Trend: slope={coeffs[0]:.2f}')
            ax.legend()
        ax = axes[1]
        y_lo = (sc_grp['fps_median'] - sc_grp['fps_p25']).clip(lower=0)
        y_hi = (sc_grp['fps_p75'] - sc_grp['fps_median']).clip(lower=0)
        ax.errorbar(sc_grp['sphere_count']/1e6, sc_grp['fps_median'], yerr=[y_lo, y_hi],
                    marker='s', capsize=5, lw=2, color='darkorange')
        ax.set_xscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('Median FPS')
        ax.set_title('FPS with interquartile range (P25–P75)')
        ax.grid(True, which='both', alpha=0.3)
        plt.tight_layout()
        show_and_save(fig, "batch/scaling_fps_loglog.png")


### 12g. Streaming convergence and occlusion culling effectiveness

**Requests vs time:** how requested blocks decay as the pool fills.

**Post-occlusion / visible ratio:** fraction of blocks that survive occlusion culling. A value of 0.2 means only 20% of visible blocks pass the occlusion test.


In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    n = len(order)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), squeeze=False)
    fig.suptitle('Streaming convergence: requests per batch', fontsize=14)
    for idx, bk in enumerate(order):
        r, c = divmod(idx, cols)
        ax = axes[r][c]
        sub = df_batch_all[df_batch_all['escena_id'] == bk].sort_values('batch_index')
        ax.semilogy(sub['batch_index'], sub['avg_numRequests'].clip(lower=0.1), marker='.', ms=3)
        ax.set_title(bk); ax.set_xlabel('Batch'); ax.set_ylabel('Requests (log)')
        ax.grid(True, alpha=0.3)
    for idx in range(n, rows*cols):
        r, c = divmod(idx, cols)
        axes[r][c].set_visible(False)
    plt.tight_layout()
    show_and_save(fig, "batch/streaming_requests_grid.png")

    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), squeeze=False)
    fig.suptitle('Post-occlusion / visible block ratio', fontsize=14)
    for idx, bk in enumerate(order):
        r, c = divmod(idx, cols)
        ax = axes[r][c]
        sub = df_batch_all[df_batch_all['escena_id'] == bk].sort_values('batch_index')
        ratio = sub['avg_numFiltered'] / sub['avg_numVisible'].replace(0, np.nan)
        ax.plot(sub['batch_index'], ratio, marker='.', ms=3, color='teal')
        ax.set_title(bk); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)
    for idx in range(n, rows*cols):
        r, c = divmod(idx, cols)
        axes[r][c].set_visible(False)
    plt.tight_layout()
    show_and_save(fig, "batch/occlusion_ratio_grid.png")


### 12h. Pool utilization and fraction of rasterized atoms

- **Pool utilization** = avg\_numVisible / pool\_slots: fraction of pool slots holding visible blocks.
- **Atom render fraction** = avg\_activeCount / sphere\_count: rasterized atoms vs total scene atoms.


In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    df_batch_all['pool_util_pct'] = 100.0 * df_batch_all['avg_numVisible'] / df_batch_all['pool_slots'].replace(0, np.nan)
    sns.boxplot(data=df_batch_all, x='escena_id', y='pool_util_pct', order=order, ax=axes[0], palette='coolwarm')
    axes[0].set_title('Pool utilization (visible blocks / slots)'); axes[0].tick_params(axis='x', rotation=30)
    df_batch_all['atom_fraction_pct'] = 100.0 * df_batch_all['avg_activeCount'] / df_batch_all['sphere_count'].replace(0, np.nan)
    sns.boxplot(data=df_batch_all, x='escena_id', y='atom_fraction_pct', order=order, ax=axes[1], palette='RdYlGn')
    axes[1].set_title('% rasterized atoms vs total scene spheres'); axes[1].tick_params(axis='x', rotation=30)
    plt.tight_layout()
    show_and_save(fig, "batch/pool_and_atom_fraction.png")


### 12i. Correlations between batch metrics (per scene)

Pearson correlation heatmaps for frame\_ms, FPS, visible, filtered, requests, and atoms.


In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    corr_cols = ['avg_frame_ms','avg_fps','avg_numVisible','avg_numFiltered','avg_numRequests','avg_activeCount']
    n = len(order)
    cols_per_row = min(3, n)
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(6*cols_per_row, 5*rows), squeeze=False)
    fig.suptitle('Pearson correlations between batch metrics', fontsize=14)
    for idx, bk in enumerate(order):
        r, c = divmod(idx, cols_per_row)
        ax = axes[r][c]
        sub = df_batch_all[df_batch_all['escena_id'] == bk][corr_cols].dropna()
        if len(sub) >= 3:
            corr = sub.corr()
            sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax, cbar=False,
                        xticklabels=[c.replace('avg_','') for c in corr_cols],
                        yticklabels=[c.replace('avg_','') for c in corr_cols])
        ax.set_title(bk)
    for idx in range(n, rows * cols_per_row):
        r, c = divmod(idx, cols_per_row)
        axes[r][c].set_visible(False)
    plt.tight_layout()
    show_and_save(fig, "batch/correlation_grid.png")


### 12j. FPS and frame time distributions per scene

Histograms with KDE to show performance variability.


In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    n = len(order)
    fig, axes = plt.subplots(2, n, figsize=(5*n, 8), squeeze=False)
    fig.suptitle('Performance distributions per scene', fontsize=14)
    for idx, bk in enumerate(order):
        sub = df_batch_all[df_batch_all['escena_id'] == bk]
        axes[0, idx].hist(sub['avg_fps'], bins=40, alpha=0.7, color='steelblue', edgecolor='white')
        axes[0, idx].set_title(f'{bk} - FPS')
        axes[1, idx].hist(sub['avg_frame_ms'], bins=40, alpha=0.7, color='coral', edgecolor='white')
        axes[1, idx].set_title(f'{bk} - Frame time')
    plt.tight_layout()
    show_and_save(fig, "batch/performance_distributions.png")


### 12k. Preprocess scaling: algorithmic complexity

Log-log of block count and octree nodes vs spheres. On-disk block-file scaling.


In [ ]:
if PREPROCESS_CSV.exists() and 'prep_grp' in dir():
    pkg = prep_grp[prep_grp['scene_type'].str.contains('PACKAGE')].copy()
    if len(pkg) >= 2:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        ax = axes[0]
        ax.plot(pkg['sphere_count']/1e6, pkg['block_count'], marker='o', color='navy')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('Blocks')
        ax.set_title('Generated blocks vs entity count'); ax.grid(True, alpha=0.3)
        ax = axes[1]
        ax.plot(pkg['sphere_count']/1e6, pkg['octree_nodes'], marker='s', color='darkred')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('Octree nodes')
        ax.set_title('Octree nodes vs entity count'); ax.grid(True, alpha=0.3)
        ax = axes[2]
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_block_file_MB'], marker='^', color='darkgreen')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Spheres (millions)'); ax.set_ylabel('Block file (MB)')
        ax.set_title('Block file size'); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        show_and_save(fig, "preprocess/scaling_blocks_octree.png")


## 13. Cross-scene comparison (when multiple scenes are loaded)


In [ ]:
if len(dfs) >= 2:
    combined = []
    for scene_key, df in dfs.items():
        tmp = df.copy()
        tmp['scene'] = SCENES[scene_key]['label']
        combined.append(tmp)
    df_all = pd.concat(combined, ignore_index=True)
    for met in ['gr_active_pct', 'gr_idle_pct', 'l1tex_hit_pct', 'pcie_throughput']:
        if df_all[met].dropna().empty:
            continue
        fig, ax = plt.subplots(figsize=(14, 6))
        sub = df_all[df_all['pipeline_range'].isin(focus_ranges)]
        sns.boxplot(data=sub, x='pipeline_range', y=met, hue='scene', ax=ax)
        ax.set_title(f'Comparison — {met}')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        show_and_save(fig, f"cross_scene/{_slug(met)}_comparison.png")
else:
    print('Only one scene loaded; skipping cross-scene comparison.')


## 14. Memory throughput — PCIe-bound or compute-bound?

If PCIe throughput dominates (>60%), the bottleneck is host–device transfer (streaming).  
If SM issue is high and PCIe is low → compute-bound.  
If both are low → latency-bound (waits, stalls).


In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    sub = df[df['pipeline_range'].isin(focus_ranges)].copy()
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(12, 7))
    for rng in focus_ranges:
        rng_df = sub[sub['pipeline_range'] == rng]
        if rng_df[['pcie_throughput', 'sm_issue_active']].dropna().empty:
            continue
        ax.scatter(rng_df['pcie_throughput'], rng_df['sm_issue_active'], label=rng, s=60, alpha=0.7)
    ax.set_xlabel('PCIe Throughput [%]')
    ax.set_ylabel('SM Issue Active [%]')
    ax.set_title(f"{cfg['label']} — Memory-bound vs compute-bound?")
    ax.legend(fontsize=8)
    plt.tight_layout()
    show_and_save(fig, f"nsight/memory_bound/{scene_key}.png")


## 15. Executive summary


In [90]:
print('=' * 80)
print('EXECUTIVE SUMMARY — OOC Pipeline Analysis')
print('=' * 80)

for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    print(f"\n{'─' * 60}")
    print(f"Scene: {cfg['label']}")
    print(f"Samples: {df['sample'].nunique()} | Rings: {df['ring'].nunique()}")

    grp = df.groupby('pipeline_range')

    # Top duration
    dur = grp['rel_frame_duration'].mean().sort_values(ascending=False)
    print(f"\nMost expensive stages (mean frame fraction):")
    for rng, val in dur.head(5).items():
        print(f"  {rng:40s} {val:.4f} ({val*100:.1f}%)")

    # Idle
    idle = grp['gr_idle_pct'].mean().sort_values(ascending=False)
    high_idle = idle[idle > 50]
    if not high_idle.empty:
        print(f"\nStages with high GPU idle (>50%):")
        for rng, val in high_idle.items():
            print(f"  {rng:40s} {val:.1f}%")

    # PCIe
    pcie = grp['pcie_throughput'].mean().sort_values(ascending=False)
    high_pcie = pcie[pcie > 30]
    if not high_pcie.empty:
        print(f"\nStages with high PCIe throughput (>30%, possible memory bottleneck):")
        for rng, val in high_pcie.items():
            print(f"  {rng:40s} {val:.1f}%")

    # Cache
    cache = grp['l1tex_hit_pct'].mean().sort_values(ascending=True)
    low_cache = cache[cache < 60]
    if not low_cache.empty:
        print(f"\nStages with low L1TEX hit rate (<60%):")
        for rng, val in low_cache.items():
            print(f"  {rng:40s} {val:.1f}%")

if BATCH_CSVS and 'df_batch_all' in dir():
    print(f"\n{'─' * 60}")
    print("Stats Batch (runtime):")
    if 'avg_fps' in df_batch_all.columns:
        print(f"  Mean FPS:  {df_batch_all['avg_fps'].mean():.1f} (min {df_batch_all['avg_fps'].min():.1f}, max {df_batch_all['avg_fps'].max():.1f})")
    if 'avg_frame_ms' in df_batch_all.columns:
        print(f"  Frame time:    {df_batch_all['avg_frame_ms'].mean():.2f} ms")
    if 'avg_activeCount' in df_batch_all.columns:
        print(f"  Mean active atoms: {df_batch_all['avg_activeCount'].mean():.0f}")

if PREPROCESS_CSV.exists() and 'prep_grp' in dir():
    print(f"\n{'─' * 60}")
    print("Preprocess:")
    if 'avg_ms' in prep_grp.columns:
        print(f"  Total time: {prep_grp['avg_ms'].iloc[-1]:.1f} ms")
    if 'avg_throughput' in prep_grp.columns:
        print(f"  Throughput:   {prep_grp['avg_throughput'].iloc[-1]:.0f} atoms/ms")

print(f"\n{'=' * 80}")


EXECUTIVE SUMMARY — OOC Pipeline Analysis

────────────────────────────────────────────────────────────
Scene: PACKAGE_SCENE ~10M spheres
Samples: 15 | Rings: 5

Most expensive stages (mean frame fraction):
  Build Active Atom List                   0.4503 (45.0%)
  Sphere Raster OOC                        0.3797 (38.0%)
  HiZ Downsample                           0.0834 (8.3%)
  Swap Window                              0.0323 (3.2%)
  Request Generation                       0.0283 (2.8%)

Stages with high GPU idle (>50%):
  Blit Framebuffer                         80.2%
  Octree BFS Frustum Culling               72.3%
  Compute Block Depth+Area                 66.7%
  Thrust Sort (Depth)                      54.4%
  Screen Clear                             52.0%

Stages with high PCIe throughput (>30%, possible memory bottleneck):
  Screen Clear                             34.1%

Stages with low L1TEX hit rate (<60%):
  Sphere Raster OOC                        18.1%
  Request Generati